# Watch parser, версия 2: коды ошибок

Что изменено относительно v1:
- 240 успешных + 240 ошибочных примеров (50/50) вместо 216/54;
- негативные примеры построены из успешных: убрать тикер, убрать период,
  оставить только сводку, подменить тикер на неизвестный, поставить
  «30 секунд», добавить третий период;
- train/valid — 432/48; eval тот же самый, чтобы сравнение было честным.

Загрузите **этот ноутбук** в Colab, включите T4 и выполните ячейки по порядку.
Ноутбук попросит `watch_parser.zip`.

In [ ]:
!pip -q install -U "transformers>=4.56" "peft>=0.13" accelerate matplotlib
!pip -q uninstall -y torchao
# После этой ячейки: Среда выполнения → Перезапустить сеанс, затем выполнять со следующей.

In [ ]:
import glob
import os
import subprocess

from google.colab import files

os.chdir('/content')
project = '/content/finetuning/watch_parser'
v2_train = f'{project}/data/v2/train.jsonl'

if not os.path.exists(v2_train):
    print('Выберите файл watch_parser.zip')
    files.upload()
    zips = sorted(glob.glob('/content/*.zip'))
    if not zips:
        raise RuntimeError('zip не найден в /content')
    subprocess.run(['unzip', '-qo', zips[-1], '-d', '/content'], check=True)

os.chdir(project)
print('проект:', os.getcwd())
print('v2 train:', os.path.exists(v2_train))

In [ ]:
!python scripts/check_dataset.py --train data/v2/train.jsonl --valid data/v2/valid.jsonl --eval data/eval.jsonl --report data/v2/check_report.json
!python scripts/check_bot_compat.py --train data/v2/train.jsonl --valid data/v2/valid.jsonl --eval data/eval.jsonl

In [ ]:
!python training/train_lora.py \
  --train data/v2/train.jsonl \
  --valid data/v2/valid.jsonl \
  --out /content/adapter_watch_json_v2 \
  --run-dir /content/run_watch_json_v2

In [ ]:
# База с промптом уже есть в results/baseline/base_prompt.json; если нет — прогоняем.
import os

if not os.path.exists('results/baseline/base_prompt.json'):
    !python scripts/eval_model.py --prompt-mode fewshot --out results/baseline/base_prompt.json

# Дообученная v2
!python scripts/eval_model.py \
  --adapter /content/adapter_watch_json_v2 \
  --prompt-mode fewshot \
  --out results/v2/tuned_prompt.json

In [ ]:
!python scripts/compare_v2.py \
  --base-prompt results/baseline/base_prompt.json \
  --tuned-v1 results/v1/tuned_prompt.json \
  --tuned-v2 results/v2/tuned_prompt.json \
  --out results/v2/compare.md

from IPython.display import Markdown, display

display(Markdown(open('results/v2/compare.md', encoding='utf-8').read()))

In [ ]:
!cd /content && zip -qr watch_results_v2.zip \
  finetuning/watch_parser/results \
  run_watch_json_v2 \
  adapter_watch_json_v2

from google.colab import files

files.download('/content/watch_results_v2.zip')